# FRTB SA — End-to-End Pipeline Demo

Demonstrates the full FRTB Standardised Approach capital calculation across all five risk classes:
**GIRR · CSR · Equity · FX · Commodity**

```
Trading_Portfolio
    └── FRTB_Sensitivity_Engine   ← calls QRE, applies is_long sign
          ├── SA_GIRR_Calculator
          ├── SA_CSR_Delta_Calculator
          ├── SA_Equity_Delta_Calculator
          ├── SA_FX_Delta_Calculator
          └── SA_Commodity_Delta_Calculator
                └── FRTB_SA_Result
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from banking_risk.frtb.stubs             import build_example_portfolio
from banking_risk.frtb.sensitivity_engine import FRTB_Sensitivity_Engine
from banking_risk.frtb.girr.delta        import SA_GIRR_Calculator
from banking_risk.frtb.csr.delta         import SA_CSR_Delta_Calculator
from banking_risk.frtb.equity.delta      import SA_Equity_Delta_Calculator
from banking_risk.frtb.fx.delta          import SA_FX_Delta_Calculator
from banking_risk.frtb.commodity.delta   import SA_Commodity_Delta_Calculator
from banking_risk.frtb.sa                import FRTB_SA
from banking_risk.frtb.vertex_mapping    import (
    FRTB_GIRR_LABELS, FRTB_CSR_LABELS, FRTB_COMMODITY_LABELS
)
from banking_risk.utils.reporting import Dark_Style

Dark_Style().apply()
p = Dark_Style().palette

## 1. Portfolio

In [ ]:
portfolio, curve = build_example_portfolio()

pd.DataFrame([
    {
        'name'      : ti.name,
        'risk_class': list(ti.risk_classes)[0].value,
        'currency'  : ti.currency,
        'direction' : 'long' if ti.is_long else 'short',
        'bucket'    : ti.csr_bucket or ti.equity_bucket or ti.commodity_bucket or '—',
        'issuer'    : ti.issuer or '—',
    }
    for ti in portfolio.instruments()
]).set_index('name')

## 2. Raw Sensitivities

In [ ]:
engine = FRTB_Sensitivity_Engine(portfolio, curve)

girr_d = engine.girr_delta()
csr_d  = engine.csr_delta()
eq_d   = engine.equity_delta()
fx_d   = engine.fx_delta()
comm_d = engine.commodity_delta()

print('GIRR delta — EUR (currency per bp):')
display(pd.Series(girr_d['EUR'], index=FRTB_GIRR_LABELS).rename('DV01').to_frame().T)

print('\nCSR delta — by bucket (currency per bp):')
display(pd.DataFrame(
    {f'bucket {b}': pd.Series(arr, index=FRTB_CSR_LABELS) for b, arr in csr_d.items()}
).T)

print('\nEquity delta — by bucket:')
display(pd.DataFrame([
    {'bucket': b, 'names': len(v), 'net_delta': sum(v)} for b, v in eq_d.items()
]).set_index('bucket'))

print('\nFX delta — by pair:')
display(pd.Series(fx_d).rename('spot delta').to_frame())

print('\nCommodity delta — by bucket (currency per 1%):')
display(pd.DataFrame(
    {f'bucket {b}': pd.Series(arr, index=FRTB_COMMODITY_LABELS) for b, arr in comm_d.items()}
).T)

## 3. SA Capital Per Risk Class

In [ ]:
girr_r = SA_GIRR_Calculator().compute(girr_d)
csr_r  = SA_CSR_Delta_Calculator().compute(csr_d)
eq_r   = SA_Equity_Delta_Calculator().compute(eq_d)
fx_r   = SA_FX_Delta_Calculator().compute(fx_d)
comm_r = SA_Commodity_Delta_Calculator().compute(comm_d)

for label, result in [
    ('GIRR delta  ', girr_r),
    ('CSR delta   ', csr_r),
    ('Equity delta', eq_r),
    ('FX delta    ', fx_r),
    ('Commodity   ', comm_r),
]:
    print(f'{label} capital: {result.capital:>12,.2f}')

In [ ]:
print('GIRR detail:')
display(girr_r.to_table())

print('\nCSR detail:')
display(csr_r.to_table())

print('\nEquity detail:')
display(eq_r.to_table())

print('\nFX detail:')
display(fx_r.to_table())

print('\nCommodity detail:')
display(comm_r.to_table())

## 4. Total FRTB SA Capital

In [ ]:
frtb = FRTB_SA(portfolio, curve)

print(f'FRTB SA Total Capital: {frtb.total:,.2f}')
display(
    frtb.to_table()
        .style
        .format('{:,.2f}')
        .set_properties(**{'text-align': 'right'})
)

In [ ]:
frtb.plot()

## 5. Drill-Down — GIRR and CSR

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# GIRR: risk-weighted sensitivity per vertex
ax = axes[0]
for ccy, ws in frtb.girr.delta.ws.items():
    ax.bar(FRTB_GIRR_LABELS, ws.values, alpha=0.85, label=ccy)
ax.axhline(0, color='white', linewidth=0.5, alpha=0.4)
ax.set_title('GIRR — WS per vertex', color=p['text_title'])
ax.set_ylabel('Risk-weighted sensitivity')
ax.tick_params(axis='x', rotation=45)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend()
ax.grid(axis='y', alpha=0.3)

# CSR: within-bucket capital K
ax = axes[1]
buckets = list(frtb.csr.delta.K.keys())
K_vals  = [frtb.csr.delta.K[b] for b in buckets]
ax.bar([str(b) for b in buckets], K_vals, color=p['cyan'], alpha=0.85)
ax.set_title('CSR — K per bucket', color=p['text_title'])
ax.set_ylabel('Capital K')
ax.set_xlabel('Bucket')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.grid(axis='y', alpha=0.3)

fig.tight_layout()
plt.show()

## 6. Netting — Long vs Short Within a Bucket

Equity bucket 5 (large cap DM consumer): long AAPL +450k, short AMZN −380k.

In [ ]:
b5 = eq_d.get(5, [])

gross_long  = sum(s for s in b5 if s > 0)
gross_short = sum(s for s in b5 if s < 0)
net         = sum(b5)

k_gross = SA_Equity_Delta_Calculator().compute({5: [gross_long]}).capital
k_net   = SA_Equity_Delta_Calculator().compute({5: b5}).capital

pd.DataFrame([
    {'metric': 'Gross long',               'value': gross_long},
    {'metric': 'Gross short',              'value': gross_short},
    {'metric': 'Net',                      'value': net},
    {'metric': 'Capital (gross long only)','value': k_gross},
    {'metric': 'Capital (net)',            'value': k_net},
    {'metric': 'Netting benefit',          'value': k_gross - k_net},
]).set_index('metric').style.format('{:,.2f}')